# Практика обучения модели: прогноз оттока клиентов телеком-компании

**Курс:** «Специалист по искусственному интеллекту» (aiml-5)
**Занятие:** «Классический ML: обучение моделей в scikit-learn»
**Задание:** «Практика обучения модели» — полный цикл обучения модели в scikit-learn
**Дата выполнения:** 26 октября 2026 г.
**Среда:** Python 3.14, pandas 3.0.6, NumPy 2.5.2, scikit-learn 1.9.1, Jupyter Notebook
**Данные:** открытый набор Telco Customer Churn (IBM Sample Data Sets), 7 043 строки, 21 столбец
**Проверка:** ноутбук выполнен целиком через *Restart & Run All* — ошибок нет.

Ноутбук последовательно проходит все шаги задания: выбор и описание данных → предобработка
(пропуски, выбросы, кодирование категорий) → разделение на train/test → обучение
`LogisticRegression` методами `.fit()` и `.predict()` → расчёт метрик → проверка
вырожденности → вывод о пригодности модели. Перед каждой ячейкой с кодом есть пояснение.

## 1. Постановка задачи и выбор данных

### 1.1. Бизнес-задача

Телеком-компании дешевле удержать клиента, чем привлечь нового. Поэтому важно заранее
находить абонентов, склонных к оттоку: с ними можно работать адресно — предложить скидку,
сменить тариф, подключить дополнительные услуги.

### 1.2. ML-задача

Бинарная классификация: по характеристикам клиента предсказать целевую переменную
**`Churn`** — уйдёт клиент (`Yes`) или останется (`No`).

### 1.3. Набор данных

- **Источник:** Telco Customer Churn, IBM Sample Data Sets —
  [файл `Telco-Customer-Churn.csv`](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv).
  Локальная копия лежит в `data/Telco-Customer-Churn.csv`.
- **Объём:** 7 043 клиента, 21 столбец.
- **Признаки:**
  - демография — `gender`, `SeniorCitizen`, `Partner`, `Dependents`;
  - услуги — `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`,
    `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`;
  - договор и оплата — `Contract`, `PaperlessBilling`, `PaymentMethod`;
  - числовые показатели — `tenure` (стаж в месяцах), `MonthlyCharges` (ежемесячный платёж),
    `TotalCharges` (сумма всех платежей).
- **Целевая переменная:** `Churn` (`Yes` / `No`).
- **Почему этот датасет:** он открытый и не содержит персональных данных (имён, адресов,
  телефонов; `customerID` — случайный идентификатор, его мы удалим). В нём одновременно есть
  пропуски, категориальные и числовые признаки, то есть все этапы предобработки из задания
  становятся осмысленными.

### 1.4. Метрики

Базовой метрикой возьмём **accuracy** (доля верных ответов), как требует задание. Но классы
несбалансированы — уходит примерно каждый четвёртый клиент, — поэтому дополнительно
посчитаем **precision, recall, F1 и ROC-AUC** и сравним модель с «глупым» baseline.

## 2. Импорт библиотек

Импортируем pandas и NumPy для работы с таблицами и вычислений, модули scikit-learn для
разделения выборки, масштабирования, обучения модели и расчёта метрик. Константа
`RANDOM_STATE = 42` фиксирует генератор случайных чисел, чтобы результат воспроизводился
при повторном запуске.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

print("pandas       ", pd.__version__)
print("numpy        ", np.__version__)
print("scikit-learn ", sklearn.__version__)

pandas        3.0.6
numpy         2.5.2
scikit-learn  1.9.1


## 3. Загрузка данных

Читаем CSV в `DataFrame` функцией `pd.read_csv()` и смотрим на первые пять строк, чтобы
убедиться, что разделитель определился верно, а значения выглядят ожидаемо.

In [2]:
DATA_PATH = "data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)
print(f"Размер таблицы: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head()

Размер таблицы: 7043 строк, 21 столбцов


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Метод `info()` показывает типы столбцов и количество непустых значений. Обратите внимание:
у `TotalCharges` тип `object` (строка), хотя это сумма платежей — значит, колонку придётся
приводить к числу.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

`info()` отчитался о 7 043 непустых значениях в каждом столбце, но это не гарантия
отсутствия пропусков: в `TotalCharges` часть значений записана пробелом, а не пустой
ячейкой, поэтому `isna()` их пока не видит. Это типичная ловушка реальных данных.

Посмотрим на баланс целевой переменной.

In [4]:
print("Абсолютные значения:")
print(df["Churn"].value_counts())
print()
print("Доли, %:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

Абсолютные значения:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Доли, %:
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


Классы несбалансированы: 73,46% клиентов остаются, 26,54% уходят. Это значит, что
accuracy 73,5% достигается моделью, которая всегда отвечает «не уйдёт». Поэтому одной
accuracy для вывода недостаточно — нужен baseline и метрики по классу «уйдёт».

## 4. Предобработка данных

### 4.1. Пропуски, дубликаты и типы

Проверяем явные пропуски (`isna()`), дубликаты строк и повторы `customerID`, затем
преобразуем `TotalCharges` в число: всё, что не является числом (пробелы), станет `NaN`
и станет видимым.

In [5]:
na_counts = df.isna().sum()
print("Явные пропуски NaN по столбцам (ненулевые):")
print(na_counts[na_counts > 0] if (na_counts > 0).any() else "нет")
print()
print("Дубликаты строк:", int(df.duplicated().sum()))
print("Повторы customerID:", int(df["customerID"].duplicated().sum()))

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
missing_total = df["TotalCharges"].isna()
print()
print("TotalCharges после to_numeric — пропусков:", int(missing_total.sum()))
print(df.loc[missing_total, ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

Явные пропуски NaN по столбцам (ненулевые):
нет

Дубликаты строк: 0
Повторы customerID: 0

TotalCharges после to_numeric — пропусков: 11
      tenure  MonthlyCharges  TotalCharges Churn
488        0           52.55           NaN    No
753        0           20.25           NaN    No
936        0           80.85           NaN    No
1082       0           25.75           NaN    No
1340       0           56.05           NaN    No
3331       0           19.85           NaN    No
3826       0           25.35           NaN    No
4380       0           20.00           NaN    No
5218       0           19.70           NaN    No
6670       0           73.35           NaN    No
6754       0           61.90           NaN    No


Дубликатов и повторяющихся идентификаторов нет. После приведения к числу нашлось **11
пропусков** в `TotalCharges` (0,16% строк), и у всех этих клиентов `tenure = 0` — это
новые абоненты, которые ещё не получили ни одного счёта.

Как поступить с пропусками:

- **заполнить медианой** (~1 394) — но тогда у клиента с нулевым стажем появится
  «накопленный платёж» в 1 394, чего не бывает. Это ловушка: формально пропуск закрыт,
  но наблюдение искажено;
- **удалить строки** — допустимо, но мы потеряем часть данных и не сможем использовать
  этих клиентов при обучении;
- **заполнить нулём** — `tenure = 0` означает, что клиент ещё не пользовался услугами
  и накопленных платежей у него действительно нет, поэтому 0 — логичное значение.

Выбираем заполнение нулём: оно не создаёт искажённого наблюдения и сохраняет все 7 043
строки. Решение фиксируем явно, чтобы его можно было воспроизвести.

In [6]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print(f"Строк в таблице:     {len(df)}")
print(f"Заполнено нулём:     {int(missing_total.sum())}")
print("Осталось пропусков:", int(df.isna().sum().sum()))

Строк в таблице:     7043
Заполнено нулём:     11
Осталось пропусков: 0


### 4.2. Выбросы

Проверим числовые признаки правилом 1,5 интерквартильного размаха (IQR): значения ниже
`Q1 − 1,5·IQR` или выше `Q3 + 1,5·IQR` считаются выбросами. Дополнительно посмотрим
описательные статистики.

In [7]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

outlier_report = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < lower) | (df[col] > upper)).sum())
    outlier_report.append(
        {
            "признак": col,
            "Q1": round(q1, 2),
            "Q3": round(q3, 2),
            "нижняя граница": round(lower, 2),
            "верхняя граница": round(upper, 2),
            "выбросов": n_out,
        }
    )

pd.DataFrame(outlier_report)

,признак,Q1,Q3,нижняя граница,верхняя граница,выбросов
0,tenure,9.00,55.00,-60.00,124.00,0
1,MonthlyCharges,35.50,89.85,-46.02,171.38,0
2,TotalCharges,398.55,3786.60,-4683.52,8868.67,0


Дополним картину описательными статистиками: они показывают реальный диапазон значений
и помогают понять, насколько границы IQR соответствуют данным.

In [8]:
df[numeric_cols].describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80


По правилу 1,5·IQR выбросов нет ни в одном числовом признаке: границы шире реального
диапазона значений. Большие суммы `TotalCharges` (до 8 684) — это не ошибки, а клиенты
с длинным стажем и дорогим тарифом, поэтому удалять или «обрезать» их нельзя: мы потеряли
бы платёжеспособных клиентов. Оставляем данные как есть.

### 4.3. Целевая переменная и матрица признаков

Кодируем `Churn` в 0/1: `No → 0`, `Yes → 1`. Колонку `customerID` удаляем — это
идентификатор, а не признак: у него 7 043 уникальных значения, и модель могла бы
«запоминать» клиентов вместо того, чтобы искать закономерности. Все остальные колонки
складываем в `X`, целевую — в `y`.

In [9]:
y = df["Churn"].map({"No": 0, "Yes": 1})
X = df.drop(columns=["customerID", "Churn"])

print("y — распределение:", y.value_counts().sort_index().to_dict())
print("X — размер матрицы признаков:", X.shape)
print(f"Доля оттока: {y.mean() * 100:.2f}%")

y — распределение: {0: 5174, 1: 1869}
X — размер матрицы признаков: (7043, 19)
Доля оттока: 26.54%


### 4.4. Разделение на обучающую и тестовую выборки

Делим данные в пропорции 80/20: 80% — обучение, 20% — проверка на данных, которые модель
не видела. Параметр `random_state=42` фиксирует разбиение, а `stratify=y` сохраняет долю
оттока одинаковой в обеих частях — иначе при дисбалансе можно случайно получить очень
разные по составу выборки.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
print(f"Доля оттока в train: {y_train.mean() * 100:.2f}%")
print(f"Доля оттока в test:  {y_test.mean() * 100:.2f}%")

X_train: (5634, 19), X_test: (1409, 19)
y_train: (5634,), y_test: (1409,)
Доля оттока в train: 26.54%
Доля оттока в test:  26.54%


### 4.5. Масштабирование числовых признаков

`tenure`, `MonthlyCharges` и `TotalCharges` имеют разный масштаб (месяцы против тысяч
рублей). Логистическая регрессия с оптимизатором `lbfgs` на несмасштабированных данных
сходится медленно и может выдать предупреждение о недостижении сходимости. Поэтому
стандартизируем числовые признаки: вычитаем среднее и делим на стандартное отклонение.

Важно: `StandardScaler` обучаем **только на train** (`.fit_transform`), а к тесту
применяем уже найденные параметры (`.transform`). Иначе среднее и разброс «подсмотрят»
тестовую выборку — это утечка данных.

In [11]:
scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train[numeric_cols].describe().round(3)

,tenure,MonthlyCharges,TotalCharges
count,5634.000,5634.000,5634.000
mean,-0.000,-0.000,0.000
std,1.000,1.000,1.000
min,-1.322,-1.544,-1.009
25%,-0.956,-0.971,-0.832
50%,-0.142,0.185,-0.397
75%,0.916,0.832,0.674
max,1.608,1.786,2.802


### 4.6. Кодирование категориальных признаков

Текстовые колонки нужно превратить в числа. Используем `pd.get_dummies()` (one-hot
кодирование), а не `LabelEncoder`: почти все категориальные признаки здесь **номинальные**
(`Contract`, `PaymentMethod`, `InternetService` и т. п.), между их значениями нет порядка.
`LabelEncoder` присвоил бы им числа 0, 1, 2, и модель восприняла бы это как «больше/меньше»,
что неверно.

Параметр `drop_first=True` убирает первый уровень каждой категории: информация о нём
восстанавливается из остальных, а лишняя колонка создаёт мультиколлинеарность («ловушка
фиктивных переменных»).

Кодирование выполняем **после** разделения и отдельно для train и test. Затем выравниваем
набор столбцов теста по train (`reindex`): если в тесте какой-то уровень не встретился,
колонка получит 0, и размерности совпадут.

In [12]:
X_train = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, drop_first=True, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Признаков после кодирования:", X_train.shape[1])
print("Формы train/test:", X_train.shape, X_test.shape)
print("Столбцы совпадают:", list(X_train.columns) == list(X_test.columns))
print("Пропусков в train/test:", int(X_train.isna().sum().sum()), int(X_test.isna().sum().sum()))
print("Типы значений:", X_train.dtypes.value_counts().to_dict())

Признаков после кодирования: 30
Формы train/test: (5634, 30) (1409, 30)
Столбцы совпадают: True
Пропусков в train/test: 0 0
Типы значений: {dtype('int64'): 27, dtype('float64'): 3}


Итог предобработки: 30 числовых признаков, пропусков нет, категории закодированы one-hot,
числовые стандартизированы. Теперь данные готовы к обучению.

## 5. Обучение модели

### 5.1. Выбор и создание модели

Задача — бинарная классификация, поэтому берём **логистическую регрессию**
(`LogisticRegression`). Она подходит для базовой модели: быстро обучается, устойчива на
небольших табличных данных и интерпретируема — по коэффициентам видно направление влияния
каждого признака. Параметр `max_iter=1000` даёт оптимизатору достаточный запас итераций
(фактически нужно около 40), `random_state` фиксирует воспроизводимость.

Создаём объект модели, но пока не обучаем.

In [13]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

### 5.2. Обучение методом `.fit()`

Передаём модели только обучающую часть — `X_train` и `y_train`. Метод `.fit()` подбирает
коэффициенты так, чтобы минимизировать ошибку логистической функции потерь. Тестовая
выборка в обучении не участвует.

In [14]:
model.fit(X_train, y_train)

print("Модель обучена")
print("Число признаков:", model.n_features_in_)
print("Итераций оптимизатору потребовалось:", int(model.n_iter_[0]))

Модель обучена
Число признаков: 30
Итераций оптимизатору потребовалось: 40


### 5.3. Предсказания методом `.predict()`

Метод `.predict()` возвращает класс (0 или 1) для каждого объекта теста. Дополнительно
берём `.predict_proba()[:, 1]` — вероятность оттока: она понадобится для ROC-AUC и позже
для подбора порога.

In [15]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Получено предсказаний:", len(y_pred))
print("Первые 10 прогнозов:", y_pred[:10])
print("Первые 10 вероятностей:", np.round(y_proba[:10], 3))

Получено предсказаний: 1409
Первые 10 прогнозов: [0 1 0 0 0 1 0 0 0 0]
Первые 10 вероятностей: [0.046 0.684 0.06  0.399 0.022 0.604 0.446 0.129 0.003 0.396]


## 6. Оценка качества

### 6.1. Accuracy

`accuracy_score` — доля объектов, для которых прогноз совпал с фактом. Считаем её на
тестовой выборке: `y_test` — реальные ответы, `y_pred` — прогнозы модели.

In [16]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy на тестовой выборке: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Верных ответов: {int(accuracy * len(y_test))} из {len(y_test)}")

Accuracy на тестовой выборке: 0.8062 (80.62%)
Верных ответов: 1136 из 1409


### 6.2. Дополнительные метрики

Accuracy не показывает, как модель ведёт себя с меньшим классом. Считаем:

- **precision** — из тех, кого модель назвала уходящими, какая доля действительно ушла;
- **recall** — какую долю реально ушедших клиентов модель нашла;
- **F1** — среднее гармоническое precision и recall;
- **ROC-AUC** — насколько хорошо модель ранжирует клиентов по вероятности оттока
  независимо от выбранного порога (0,5 — случайно, 1,0 — идеально).

Матрица ошибок показывает абсолютные числа: TN, FP, FN, TP.

In [17]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print()
print("Матрица ошибок [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))
print()
print("Отчёт по классам:")
print(classification_report(y_test, y_pred, target_names=["останется (0)", "уйдёт (1)"]))

Precision: 0.6593
Recall:    0.5588
F1:        0.6049
ROC-AUC:   0.8422

Матрица ошибок [[TN, FP], [FN, TP]]:
[[927 108]
 [165 209]]

Отчёт по классам:


               precision    recall  f1-score   support

останется (0)       0.85      0.90      0.87      1035
    уйдёт (1)       0.66      0.56      0.60       374

     accuracy                           0.81      1409
    macro avg       0.75      0.73      0.74      1409
 weighted avg       0.80      0.81      0.80      1409



### 6.3. Сравнение факта и прогноза

Выводим первые 15 объектов теста рядом: факт, прогноз и вероятность оттока. Так видно,
на каких клиентах модель уверена, а где ошибается и насколько уверенно.

In [18]:
comparison = pd.DataFrame(
    {
        "y_test (факт)": y_test.head(15).values,
        "y_pred (прогноз)": y_pred[:15],
        "вероятность оттока": np.round(y_proba[:15], 3),
    }
)
comparison["совпало"] = comparison["y_test (факт)"] == comparison["y_pred (прогноз)"]
comparison

,y_test (факт),y_pred (прогноз),вероятность оттока,совпало
0,0,0,0.046,True
1,0,1,0.684,False
2,0,0,0.060,True
3,0,0,0.399,True
4,0,0,0.022,True
5,0,1,0.604,False
6,0,0,0.446,True
7,0,0,0.129,True
8,0,0,0.003,True
9,1,0,0.396,False


### 6.4. Проверка на вырожденное поведение

Главная проверка из задания: модель не должна выдавать одно и то же значение для всех
объектов. Смотрим распределение прогнозов и число уникальных классов. Для контраста обучаем
`DummyClassifier(strategy="most_frequent")` — он всегда предсказывает самый частый класс
(«не уйдёт») и показывает, какую accuracy даёт полностью вырожденная модель.

In [19]:
pred_distribution = pd.Series(y_pred).value_counts().sort_index()
print("Распределение прогнозов модели:")
print(pred_distribution.to_dict())
print("Уникальные значения в y_pred:", np.unique(y_pred))
print()

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_accuracy = accuracy_score(y_test, dummy_pred)

print("DummyClassifier (всегда «не уйдёт»):")
print("  уникальные значения:", np.unique(dummy_pred))
print(f"  accuracy: {dummy_accuracy:.4f}")
print()
print(f"Прирост accuracy логистической регрессии над baseline: {accuracy - dummy_accuracy:+.4f}")

Распределение прогнозов модели:


{0: 1092, 1: 317}
Уникальные значения в y_pred: [0 1]

DummyClassifier (всегда «не уйдёт»):
  уникальные значения: [0]
  accuracy: 0.7346

Прирост accuracy логистической регрессии над baseline: +0.0717


Проверка пройдена: модель предсказывает **оба класса** — 1 092 «останется» и 317 «уйдёт»,
то есть поведение не вырождено. DummyClassifier выдаёт только нули и всё равно получает
accuracy 73,5% — именно поэтому высокая accuracy сама по себе ничего не доказывает.
Наша модель выше baseline на 7,2 п.п. и при этом реально разделяет клиентов.

### 6.5. Какие признаки сильнее всего влияют на прогноз

После стандартизации коэффициенты логистической регрессии можно сравнивать между собой:
положительный коэффициент повышает вероятность оттока, отрицательный — снижает. Смотрим
по пять самых сильных признаков в каждую сторону.

In [20]:
coefficients = pd.Series(model.coef_[0], index=X_train.columns).sort_values()

print("Снижают вероятность оттока (сильнее всего):")
print(coefficients.head(5).round(3))
print()
print("Повышают вероятность оттока (сильнее всего):")
print(coefficients.tail(5).round(3))

Снижают вероятность оттока (сильнее всего):
Contract_Two year   -1.314
tenure              -1.257
Contract_One year   -0.684
PhoneService_Yes    -0.521
MonthlyCharges      -0.467
dtype: float64

Повышают вероятность оттока (сильнее всего):
PaperlessBilling_Yes              0.374
StreamingTV_Yes                   0.375
PaymentMethod_Electronic check    0.393
TotalCharges                      0.528
InternetService_Fiber optic       1.181
dtype: float64


Интерпретация согласуется со здравым смыслом: длительный стаж (`tenure`) и долгосрочный
контракт (`Contract_Two year`) снижают риск ухода, а оптоволоконный интернет
(`InternetService_Fiber optic`) и оплата электронным чеком (`PaymentMethod_Electronic check`)
его повышают. Коэффициенты `tenure` и `TotalCharges` смотрят в разные стороны, потому что
эти признаки сильно коррелируют между собой (стаж и накопленная сумма) — на качестве
прогноза это не сказывается, но трактовать их по отдельности нужно осторожно.

### 6.6. Вариант с учётом дисбаланса классов

Проверим, что даст `class_weight="balanced"`: он велит модели штрафовать ошибки на редком
классе сильнее, чтобы чаще находить уходящих клиентов. Это не обязательный шаг задания,
а дополнительный эксперимент — он показывает цену выбора метрики при дисбалансе.

In [21]:
model_balanced = LogisticRegression(
    max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
)
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)
y_proba_balanced = model_balanced.predict_proba(X_test)[:, 1]

balanced_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_balanced),
    "precision": precision_score(y_test, y_pred_balanced),
    "recall": recall_score(y_test, y_pred_balanced),
    "f1": f1_score(y_test, y_pred_balanced),
    "roc_auc": roc_auc_score(y_test, y_proba_balanced),
}

comparison_models = pd.DataFrame(
    {
        "LogisticRegression": {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": roc_auc,
        },
        "LogisticRegression (balanced)": balanced_metrics,
    }
).round(4)

print("Прогнозы balanced-модели:", pd.Series(y_pred_balanced).value_counts().sort_index().to_dict())
print()
comparison_models

Прогнозы balanced-модели: {0: 828, 1: 581}



,LogisticRegression,LogisticRegression (balanced)
accuracy,0.8062,0.7381
precision,0.6593,0.5043
recall,0.5588,0.7834
f1,0.6049,0.6136
roc_auc,0.8422,0.8417


Balanced-модель находит 293 ушедших клиента из 374 (recall 0,78 против 0,56), но платит
за это точностью: precision падает до 0,50, а accuracy — до 0,74, то есть почти до уровня
«глупого» baseline (73,5%). ROC-AUC у обеих моделей практически одинаков (0,84), потому что
способность ранжировать клиентов не изменилась — изменился только порог принятия решения.

Это важный вывод: при дисбалансе выбор метрики зависит от бизнес-цели. Если дороже
пропустить уходящего клиента — нужен высокий recall и balanced-вариант. Если дороже зря
беспокоить лояльных — нужен высокий precision и базовая модель. Accuracy здесь —
вспомогательная метрика.

## 7. Итоговый вывод

**Какая модель использовалась.** Логистическая регрессия
`LogisticRegression(max_iter=1000, random_state=42)` из scikit-learn. После предобработки
получилось 30 числовых признаков: числовые стандартизированы, категориальные закодированы
one-hot без первого уровня. Обучение — на 5 634 объектах, проверка — на 1 409 объектах,
которых модель не видела.

**Какие метрики получены.**

| Метрика | Значение |
|---|---|
| Accuracy | **0,8062** |
| Precision (класс «уйдёт») | 0,6593 |
| Recall (класс «уйдёт») | 0,5588 |
| F1 (класс «уйдёт») | 0,6049 |
| ROC-AUC | 0,8422 |

Матрица ошибок: 927 верно предсказанных «останется», 209 верно предсказанных «уйдёт»,
108 ложных «уйдёт» и 165 пропущенных оттоков.

**Хороший ли это результат и почему.** Для базовой модели — да. Accuracy 80,6% на 7,2
п.п. выше «глупого» baseline, который всегда отвечает «не уйдёт» (73,5%).
ROC-AUC 0,84 означает, что модель уверенно ранжирует клиентов по склонности к оттоку:
0,5 — случайное угадывание, 1,0 — идеальное разделение. Модель не вырождена: она
предсказывает оба класса (1 092 «останется» и 317 «уйдёт»), а не одно значение для всех.

При этом модель не готова к продакшену «как есть». Recall 0,56: из 374 реально ушедших
клиентов модель нашла только 209 — около 44% оттока пропущено, а именно эти клиенты и нужны
бизнесу для удержания. Precision 0,66: примерно треть клиентов, которых модель считает
уходящими, на самом деле остаются — лишние затраты на удержание. Вариант с
`class_weight="balanced"` поднимает recall до 0,78 (находит 293 из 374), но роняет precision
до 0,50 и accuracy до 0,74 — почти до уровня baseline. Это наглядно показывает, почему при
дисбалансе классов нельзя смотреть только на accuracy: метрику нужно выбирать под
бизнес-цель.

**Вывод.** Логистическая регрессия подходит как интерпретируемая базовая модель: она
значимо лучше вырожденного прогноза и хорошо разделяет классы (ROC-AUC 0,84). Для
практического применения нужно:

1. подобрать порог классификации или `class_weight` под стоимость пропущенного оттока
   и ложного срабатывания;
2. добавить кросс-валидацию и сравнить с градиентным бустингом (XGBoost / LightGBM);
3. проверить стабильность на новых периодах.

В текущем виде модель полезна как baseline и как инструмент анализа факторов оттока.

## 8. Самопроверка по критериям задания

| Критерий | Где выполнен | Статус |
|---|---|---|
| Выбран датасет, выполнена предобработка: пропуски, выбросы, кодирование категорий | разделы 1, 4.1–4.6 | ✅ |
| Данные разделены на train/test (`train_test_split`, `random_state=42`) | раздел 4.4 | ✅ |
| Модель обучена через `.fit()`, предсказания получены через `.predict()` | разделы 5.2–5.3 | ✅ |
| Рассчитана метрика качества и объяснено её значение | разделы 6.1–6.2, 7 | ✅ |
| Проверено отсутствие вырожденного поведения | раздел 6.4 | ✅ |
| Сформулирован вывод о пригодности модели | раздел 7 | ✅ |
| Ноутбук выполняется целиком без ошибок (Restart & Run All) | проверено сборщиком `tools/build_notebook.py` | ✅ |
| Перед каждой ячейкой кода есть пояснение в Markdown | все разделы | ✅ |

## 9. Как воспроизвести результат

1. Установить зависимости: `pip install pandas numpy scikit-learn notebook`.
2. Открыть `Практика_обучения_модели.ipynb` из этой папки (рядом должна лежать папка `data/`).
3. Выполнить **Restart & Run All** — notebook сам скачивать данные не будет, используется
   локальная копия `data/Telco-Customer-Churn.csv`.

Альтернативно: `python -I tools/build_notebook.py` — скрипт заново соберёт ноутбук и
выполнит все ячейки.